# Exploring `soc` and `soccfg` Parameters

This notebook connects to the QICK hardware and queries the various properties
and methods available on the `soc` (Pyro4 proxy) and `soccfg` (QickConfig) objects.

## Setup & Connection

In [1]:
cfg_file = 'sample_50_rfboard.yml'
ip = '10.108.30.23'
rfsoc_alias = 'bf1_soc'

In [2]:
import os
import numpy as np
from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
from slab_qick_calib.helpers import config

%load_ext autoreload
%autoreload 2

In [3]:
cfg_path = os.path.join(os.getcwd(), '..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'])
print(im)

soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())

{'Pyro.NameServer': <Pyro4.core.Proxy at 0x1ab349618b0; not connected; for PYRO:Pyro.NameServer@10.108.30.23:9090>, 'bf1_soc': <Pyro4.core.Proxy at 0x1ab346885f0; not connected; for PYRO:obj_ed9224d46dec403c91d22b23b484fb88@10.108.30.74:34153>}


---
# 1. `soccfg` Overview

Print the built-in description which summarises generators, readouts, tProc, clocks, etc.

In [4]:
print(soccfg)

QICK running on ZCU216, software version 0.2.360

Firmware configuration (built Sun Jun 15 22:25:25 2025):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 1, ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 65536 complex samples (6.838 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 1 is 1_228 on JHC2, or QICK box DAC port 1
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 32768 complex samples (3.419 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 2 is 2_228 on JHC1, or QICK box DAC port 2
	3:	axis_signal_ge

## 1.1 Raw configuration dictionary keys

In [5]:
raw_cfg = soccfg.get_cfg()
print('Top-level keys:', list(raw_cfg.keys()))

Top-level keys: ['board', 'sw_version', 'extra_description', 'fw_timestamp', 'rf', 'refclk_freq', 'mr_buf', 'ddr4_buf', 'gens', 'iqs', 'readouts', 'tprocs']


## 1.2 Board info

In [6]:
print('Board:        ', soccfg['board'])
print('SW version:   ', soccfg['sw_version'])
print('FW timestamp: ', soccfg['fw_timestamp'])
print('Ref clk (MHz):', soccfg['refclk_freq'])

Board:         ZCU216
SW version:    0.2.360
FW timestamp:  Sun Jun 15 22:25:25 2025
Ref clk (MHz): 245.76


## 1.3 Signal generators

In [7]:
print(f"Number of generators: {len(soccfg['gens'])}\n")
for i, gen in enumerate(soccfg['gens']):
    print(f'--- Gen ch {i} ---')
    for k, v in gen.items():
        print(f'  {k:20s}: {v}')

Number of generators: 16

--- Gen ch 0 ---
  type                : axis_signal_gen_v6
  fullpath            : axis_signal_gen_v6_0
  maxlen              : 65536
  complex_env         : True
  samps_per_clk       : 16
  maxv_scale          : 1.0
  has_dds             : True
  b_dds               : 32
  b_phase             : 32
  maxv                : 32766
  has_mixer           : False
  revision            : 4
  version             : 1.0
  dac                 : 00
  fs                  : 9584.64
  fs_mult             : 39
  fs_div              : 1
  interpolation       : 1
  f_fabric            : 599.04
  f_dds               : 9584.64
  fdds_div            : 1
  tproc_ch            : 0
--- Gen ch 1 ---
  type                : axis_signal_gen_v6
  fullpath            : axis_signal_gen_v6_1
  maxlen              : 16384
  complex_env         : True
  samps_per_clk       : 16
  maxv_scale          : 1.0
  has_dds             : True
  b_dds               : 32
  b_phase             : 32
  m

## 1.4 Readout channels

In [8]:
print(f"Number of readouts: {len(soccfg['readouts'])}\n")
for i, ro in enumerate(soccfg['readouts']):
    print(f'--- Readout ch {i} ---')
    for k, v in ro.items():
        print(f'  {k:20s}: {v}')

Number of readouts: 11

--- Readout ch 0 ---
  avg_maxlen          : 8192
  buf_maxlen          : 4096
  has_edge_counter    : True
  has_weights         : False
  trigger_type        : tport
  trigger_port        : 10
  trigger_bit         : 0
  tproc_ch            : 0
  tmux_ch             : 1
  tproc_ctrl          : 4
  adc                 : 20
  b_phase             : 32
  fs                  : 2457.6
  fs_mult             : 40
  fs_div              : 4
  decimation          : 1
  f_fabric            : 307.2
  f_dds               : 2457.6
  fdds_div            : 4
  f_output            : 307.2
  b_dds               : 32
  iq_offset           : 0.0
  has_outsel          : True
  avgbuf_fullpath     : axis_avg_buffer_0
  ro_fullpath         : axis_dyn_readout_v1_0
  avgbuf_revision     : 2
  ro_revision         : 3
  avgbuf_type         : axis_avg_buffer
  ro_type             : axis_dyn_readout_v1
  avgbuf_version      : 1.2
  ro_version          : 1.0
--- Readout ch 1 ---
  avg_maxle

## 1.5 tProcessor

In [9]:
for i, tp in enumerate(soccfg['tprocs']):
    print(f'--- tProc {i} ---')
    for k, v in tp.items():
        print(f'  {k:20s}: {v}')

--- tProc 0 ---
  type                : qick_processor
  fullpath            : qick_processor_0
  pmem_size           : 4096
  dmem_size           : 16384
  wmem_size           : 1024
  dreg_qty            : 16
  in_port_qty         : 8
  out_trig_qty        : 19
  out_dport_qty       : 1
  out_dport_dw        : 8
  out_wport_qty       : 16
  has_lfsr            : 1
  has_divider         : 1
  has_arith           : 1
  has_time_read       : 1
  has_qcom            : 0
  has_custom_periph   : 0
  has_io_ctrl         : 1
  has_ext_flag        : 0
  has_qnet            : 0
  fifo_depth          : 512
  call_depth          : 255
  debug               : 0
  clk_srcs            : {'core clock': {'source': ('dac', 2), 'f_clk': 215.04, 'src_range': [284.44444444444446, 568.8888888888889]}, 'timing clock': {'source': ('dac', 2), 'f_clk': 430.08, 'src_range': None}}
  revision            : 24
  version             : 2.0
  f_core              : 215.04
  f_time              : 430.08
  output_pins 

## 1.6 RF data converters (DACs / ADCs)

In [10]:
rf = soccfg['rf']
print('RF keys:', list(rf.keys()))

print('\n--- DACs ---')
for name, dac in rf['dacs'].items():
    print(f'  DAC {name}: {dac}')

print('\n--- ADCs ---')
for name, adc in rf['adcs'].items():
    print(f'  ADC {name}: {adc}')

RF keys: ['type', 'fullpath', 'ip_type', 'hs_adc', 'tiles', 'dacs', 'adcs', 'clk_groups', 'revision', 'version']

--- DACs ---
  DAC 00: {'index': [0, 0], 'coupling': 'AC', 'f_ref': 245.76, 'fs_mult': 39, 'fs_div': 1, 'fs': 9584.64, 'f_fabric': 599.04, 'interpolation': 1, 'datapath': 4}
  DAC 01: {'index': [0, 1], 'coupling': 'AC', 'f_ref': 245.76, 'fs_mult': 39, 'fs_div': 1, 'fs': 9584.64, 'f_fabric': 599.04, 'interpolation': 1, 'datapath': 4}
  DAC 02: {'index': [0, 2], 'coupling': 'AC', 'f_ref': 245.76, 'fs_mult': 39, 'fs_div': 1, 'fs': 9584.64, 'f_fabric': 599.04, 'interpolation': 1, 'datapath': 4}
  DAC 03: {'index': [0, 3], 'coupling': 'AC', 'f_ref': 245.76, 'fs_mult': 39, 'fs_div': 1, 'fs': 9584.64, 'f_fabric': 599.04, 'interpolation': 1, 'datapath': 4}
  DAC 10: {'index': [1, 0], 'coupling': 'AC', 'f_ref': 245.76, 'fs_mult': 56, 'fs_div': 2, 'fs': 6881.28, 'f_fabric': 430.08, 'interpolation': 4, 'datapath': 1}
  DAC 11: {'index': [1, 1], 'coupling': 'AC', 'f_ref': 245.76, 'fs_m

## 1.7 Optional blocks (IQ, DDR4, MR buffer, time taggers)

In [11]:
for key in ['iqs', 'ddr4_buf', 'mr_buf', 'time_taggers']:
    val = raw_cfg.get(key)
    if val is not None:
        print(f'{key}: {val}')
    else:
        print(f'{key}: not present')

iqs: []
ddr4_buf: {'type': 'axis_buffer_ddr_v1', 'fullpath': 'ddr4/axis_buffer_ddr_v1_0', 'burst_len': 128, 'junk_len': 401, 'junk_nt': 4, 'readouts': ['axis_avg_buffer_0', 'axis_avg_buffer_1', 'axis_avg_buffer_2', 'axis_avg_buffer_3', 'axis_avg_buffer_4', 'axis_avg_buffer_5', 'axis_avg_buffer_6', 'axis_avg_buffer_7', 'axis_avg_buffer_8', 'axis_avg_buffer_9', 'axis_avg_buffer_10'], 'revision': 3, 'version': '1.0', 'maxlen': 1073741824, 'trigger_type': 'tport', 'trigger_port': 9, 'trigger_bit': 0}
mr_buf: {'type': 'mr_buffer_et', 'fullpath': 'mr_buffer_et_0', 'maxlen': 8192, 'junk_len': 0, 'readouts': ['axis_avg_buffer_0', 'axis_avg_buffer_1', 'axis_avg_buffer_10'], 'revision': 1, 'version': '1.1', 'trigger_type': 'tport', 'trigger_port': 8, 'trigger_bit': 0}
time_taggers: not present


---
# 2. Frequency Conversion Methods

In [12]:
gen_ch = 0
ro_ch = 0
test_freq = 5000.0  # MHz

# freq <-> register
reg = soccfg.freq2reg(test_freq, gen_ch=gen_ch)
freq_back = soccfg.reg2freq(reg, gen_ch=gen_ch)
print(f'freq2reg({test_freq} MHz, gen_ch={gen_ch}) = {reg}')
print(f'reg2freq({reg}, gen_ch={gen_ch})            = {freq_back:.6f} MHz')

# ADC frequency register
reg_adc = soccfg.freq2reg_adc(test_freq, ro_ch=ro_ch)
freq_adc_back = soccfg.reg2freq_adc(reg_adc, ro_ch=ro_ch)
print(f'\nfreq2reg_adc({test_freq} MHz, ro_ch={ro_ch}) = {reg_adc}')
print(f'reg2freq_adc({reg_adc}, ro_ch={ro_ch})          = {freq_adc_back:.6f} MHz')

# Round to valid frequency for both gen and readout
matched = soccfg.adcfreq(test_freq, gen_ch=gen_ch, ro_ch=ro_ch)
print(f'\nadcfreq({test_freq}, gen={gen_ch}, ro={ro_ch}) = {matched:.6f} MHz')

freq2reg(5000.0 MHz, gen_ch=0) = 2240547009
reg2freq(2240547009, gen_ch=0)            = 5000.000001 MHz

freq2reg_adc(5000.0 MHz, ro_ch=0) = 148198741
reg2freq_adc(148198741, ro_ch=0)          = 84.800000 MHz

adcfreq(5000.0, gen=0, ro=0) = 5000.000003 MHz


## 2.1 Frequency step sizes

In [25]:
for i in range(len(soccfg['gens'])):
    gen_cfg = soccfg['gens'][i]
    step = soccfg.ch_fstep(gen_cfg)
    print(f'Gen ch {i}: freq step = {1e6*step:.6f} Hz')

print()
for i in range(len(soccfg['readouts'])):
    ro_cfg = soccfg['readouts'][i]
    step = soccfg.ch_fstep(ro_cfg)
    print(f'Readout ch {i}: freq step = {1e6*step:.6f} Hz')

Gen ch 0: freq step = 2.231598 Hz
Gen ch 1: freq step = 2.231598 Hz
Gen ch 2: freq step = 2.231598 Hz
Gen ch 3: freq step = 2.231598 Hz
Gen ch 4: freq step = 0.400543 Hz
Gen ch 5: freq step = 0.400543 Hz
Gen ch 6: freq step = 0.400543 Hz
Gen ch 7: freq step = 0.400543 Hz
Gen ch 8: freq step = 0.400543 Hz
Gen ch 9: freq step = 0.400543 Hz
Gen ch 10: freq step = 0.400543 Hz
Gen ch 11: freq step = 0.400543 Hz
Gen ch 12: freq step = 0.400543 Hz
Gen ch 13: freq step = 0.400543 Hz
Gen ch 14: freq step = 0.400543 Hz
Gen ch 15: freq step = 0.400543 Hz

Readout ch 0: freq step = 0.572205 Hz
Readout ch 1: freq step = 0.572205 Hz
Readout ch 2: freq step = 0.008941 Hz
Readout ch 3: freq step = 0.008941 Hz
Readout ch 4: freq step = 0.008941 Hz
Readout ch 5: freq step = 0.008941 Hz
Readout ch 6: freq step = 0.008941 Hz
Readout ch 7: freq step = 0.008941 Hz
Readout ch 8: freq step = 0.008941 Hz
Readout ch 9: freq step = 0.008941 Hz
Readout ch 10: freq step = 0.572205 Hz


---
# 3. Phase Conversion Methods

In [14]:
test_deg = 90.0

for ch in range(min(len(soccfg['gens']), 4)):
    reg_phase = soccfg.deg2reg(test_deg, gen_ch=ch)
    deg_back = soccfg.reg2deg(reg_phase, gen_ch=ch)
    print(f'Gen ch {ch}: deg2reg({test_deg}) = {reg_phase},  reg2deg -> {deg_back:.4f}')

Gen ch 0: deg2reg(90.0) = 1073741824,  reg2deg -> 90.0000
Gen ch 1: deg2reg(90.0) = 1073741824,  reg2deg -> 90.0000
Gen ch 2: deg2reg(90.0) = 1073741824,  reg2deg -> 90.0000
Gen ch 3: deg2reg(90.0) = 1073741824,  reg2deg -> 90.0000


---
# 4. Timing Conversion Methods

In [15]:
# tProc clock (default)
print('=== tProc / dispatcher clock ===')
print(f'1 cycle  = {soccfg.cycles2us(1):.6f} us')
print(f'1 us     = {soccfg.us2cycles(1)} cycles')

# Per-generator clocks
print('\n=== Generator fabric clocks ===')
for ch in range(min(len(soccfg['gens']), 4)):
    us_per_cycle = soccfg.cycles2us(1, gen_ch=ch)
    cycles_per_us = soccfg.us2cycles(1, gen_ch=ch)
    print(f'Gen ch {ch}: 1 cycle = {us_per_cycle:.6f} us,  1 us = {cycles_per_us} cycles')

# Per-readout clocks
print('\n=== Readout output clocks ===')
for ch in range(min(len(soccfg['readouts']), 4)):
    us_per_cycle = soccfg.cycles2us(1, ro_ch=ch)
    cycles_per_us = soccfg.us2cycles(1, ro_ch=ch)
    print(f'Readout ch {ch}: 1 cycle = {us_per_cycle:.6f} us,  1 us = {cycles_per_us} cycles')

=== tProc / dispatcher clock ===
1 cycle  = 0.002325 us
1 us     = 430 cycles

=== Generator fabric clocks ===
Gen ch 0: 1 cycle = 0.001669 us,  1 us = 599 cycles
Gen ch 1: 1 cycle = 0.001669 us,  1 us = 599 cycles
Gen ch 2: 1 cycle = 0.001669 us,  1 us = 599 cycles
Gen ch 3: 1 cycle = 0.001669 us,  1 us = 599 cycles

=== Readout output clocks ===
Readout ch 0: 1 cycle = 0.003255 us,  1 us = 307 cycles
Readout ch 1: 1 cycle = 0.003255 us,  1 us = 307 cycles
Readout ch 2: 1 cycle = 0.026042 us,  1 us = 38 cycles
Readout ch 3: 1 cycle = 0.026042 us,  1 us = 38 cycles


---
# 5. Envelope Limits

In [16]:
for ch in range(min(len(soccfg['gens']), 4)):
    print(f'Gen ch {ch}: max envelope amplitude = {soccfg.get_maxv(ch)}')

Gen ch 0: max envelope amplitude = 32766
Gen ch 1: max envelope amplitude = 32766
Gen ch 2: max envelope amplitude = 32766
Gen ch 3: max envelope amplitude = 32766


---
# 6. Per-channel detailed config

Key fields in each channel config dict: `f_fabric`, `f_output`, `f_dds`,
`b_dds`, `b_phase`, `has_mixer`, `interpolation`/`decimation`, etc.

In [17]:
# Generator channel config via _get_ch_cfg
for ch in range(min(len(soccfg['gens']), 4)):
    ch_cfg = soccfg._get_ch_cfg(gen_ch=ch)
    print(f'--- Gen ch {ch} ---')
    print(f"  f_fabric  = {ch_cfg.get('f_fabric')} MHz")
    print(f"  f_output  = {ch_cfg.get('f_output')} MHz")
    print(f"  f_dds     = {ch_cfg.get('f_dds')} MHz")
    print(f"  b_dds     = {ch_cfg.get('b_dds')} bits")
    print(f"  b_phase   = {ch_cfg.get('b_phase')} bits")
    print(f"  has_mixer = {ch_cfg.get('has_mixer')}")
    print()

--- Gen ch 0 ---
  f_fabric  = 599.04 MHz
  f_output  = None MHz
  f_dds     = 9584.64 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits
  has_mixer = False

--- Gen ch 1 ---
  f_fabric  = 599.04 MHz
  f_output  = None MHz
  f_dds     = 9584.64 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits
  has_mixer = False

--- Gen ch 2 ---
  f_fabric  = 599.04 MHz
  f_output  = None MHz
  f_dds     = 9584.64 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits
  has_mixer = False

--- Gen ch 3 ---
  f_fabric  = 599.04 MHz
  f_output  = None MHz
  f_dds     = 9584.64 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits
  has_mixer = False



In [18]:
# Readout channel config via _get_ch_cfg
for ch in range(min(len(soccfg['readouts']), 4)):
    ch_cfg = soccfg._get_ch_cfg(ro_ch=ch)
    print(f'--- Readout ch {ch} ---')
    print(f"  f_fabric  = {ch_cfg.get('f_fabric')} MHz")
    print(f"  f_output  = {ch_cfg.get('f_output')} MHz")
    print(f"  f_dds     = {ch_cfg.get('f_dds')} MHz")
    print(f"  b_dds     = {ch_cfg.get('b_dds')} bits")
    print(f"  b_phase   = {ch_cfg.get('b_phase')} bits")
    print()

--- Readout ch 0 ---
  f_fabric  = 307.2 MHz
  f_output  = 307.2 MHz
  f_dds     = 2457.6 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits

--- Readout ch 1 ---
  f_fabric  = 307.2 MHz
  f_output  = 307.2 MHz
  f_dds     = 2457.6 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits

--- Readout ch 2 ---
  f_fabric  = 307.2 MHz
  f_output  = 38.4 MHz
  f_dds     = 38.4 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits

--- Readout ch 3 ---
  f_fabric  = 307.2 MHz
  f_output  = 38.4 MHz
  f_dds     = 38.4 MHz
  b_dds     = 32 bits
  b_phase   = 32 bits



---
# 7. `soc` (Pyro4 Proxy) Methods

The `soc` object is a remote proxy to the QICK hardware.
Below we query hardware-side methods.

In [19]:
# List available remote methods
soc_methods = [m for m in dir(soc) if not m.startswith('_')]
print(f'Number of public methods/attrs: {len(soc_methods)}\n')
for m in sorted(soc_methods):
    print(f'  {m}')

Number of public methods/attrs: 85

  adcfreq
  arm_ddr4
  arm_mr
  calc_fstep
  calc_fstep_int
  calc_mixer_freq
  calc_muxgen_regs
  calc_ro_freq
  calc_ro_regs
  ch_fstep
  check_pfb_collisions
  clear_ddr4
  clear_tproc_counter
  clocks_locked
  config_avg
  config_buf
  config_clocks
  config_mux_gen
  config_mux_readout
  configure_readout
  cycles2us
  deg2int
  deg2reg
  description
  download
  dump_cfg
  enable_buf
  free
  freq2int
  freq2reg
  freq2reg_adc
  get_accumulated
  get_cfg
  get_ddr4
  get_decimated
  get_maxv
  get_mr
  get_sample_rates
  get_tproc_counter
  insert_dtbo
  int2deg
  int2freq
  is_loaded
  load_bin_program
  load_envelope
  load_ip_data
  load_mem
  load_weights
  map_signal_paths
  poll_data
  pr_download
  print_sg_mem
  read_mem
  reg2deg
  reg2freq
  reg2freq_adc
  reload_mem
  remove_dtbo
  reset
  reset_gens
  rfb_config
  rfb_disable_bias
  rfb_enable_bias
  rfb_get_bias
  rfb_set_bias
  rfb_set_gen_dc
  rfb_set_gen_filter
  rfb_set_gen_rf


## 7.1 RF board methods (if available)

These are present when the QICK board has an RF daughter board.

In [20]:
rfb_methods = [m for m in dir(soc) if m.startswith('rfb_')]
print(f'RF board methods ({len(rfb_methods)}):')
for m in sorted(rfb_methods):
    print(f'  {m}')

RF board methods (11):
  rfb_config
  rfb_disable_bias
  rfb_enable_bias
  rfb_get_bias
  rfb_set_bias
  rfb_set_gen_dc
  rfb_set_gen_filter
  rfb_set_gen_rf
  rfb_set_ro_dc
  rfb_set_ro_filter
  rfb_set_ro_rf


## 7.2 Query RF board bias (if applicable)

In [21]:
# Uncomment to read bias on a specific channel
# bias_ch = 0
# print(f'Bias ch {bias_ch}: {soc.rfb_get_bias(bias_ch)} V')

---
# 8. Sample Rates

Query DAC/ADC sampling rates from both `soccfg` (local config) and `soc` (remote hardware).

## 8.1 DAC sampling rates from `soccfg`

`fs` = actual DAC sampling rate (Msps), `f_fabric` = FPGA fabric clock,
`interpolation` = RF-DAC interpolation factor, `fs = f_ref * fs_mult / fs_div`.

In [ ]:
print('=== DAC Sampling Rates ===')
for name, dac in soccfg['rf']['dacs'].items():
    print(f"  DAC {name}: fs = {dac['fs']:.3f} Msps, "
          f"f_fabric = {dac['f_fabric']:.3f} MHz, "
          f"interpolation = {dac['interpolation']}x, "
          f"fs_mult = {dac['fs_mult']}, fs_div = {dac['fs_div']}, "
          f"f_ref = {dac['f_ref']:.2f} MHz")

## 8.2 ADC sampling rates from `soccfg`

`fs` = actual ADC sampling rate (Msps), `decimation` = RF-ADC decimation factor.

In [ ]:
print('=== ADC Sampling Rates ===')
for name, adc in soccfg['rf']['adcs'].items():
    print(f"  ADC {name}: fs = {adc['fs']:.3f} Msps, "
          f"f_fabric = {adc['f_fabric']:.3f} MHz, "
          f"decimation = {adc['decimation']}x, "
          f"fs_mult = {adc['fs_mult']}, fs_div = {adc['fs_div']}, "
          f"f_ref = {adc['f_ref']:.2f} MHz")

## 8.3 Generator effective sample rates

Each generator channel has a DAC behind it. `samps_per_clk` tells you how many
DAC samples are produced per fabric clock cycle.

In [26]:
for i, gen in enumerate(soccfg['gens']):
    dacname = gen['dac']
    dac = soccfg['rf']['dacs'][dacname]
    print(f"Gen ch {i:2d}: DAC {dacname}, "
          f"fs = {dac['fs']:.3f} Msps, "
          f"f_fabric = {gen['f_fabric']:.3f} MHz, "
          f"samps_per_clk = {gen.get('samps_per_clk', 'N/A')}, "
          f"type = {gen['type']}")

Gen ch  0: DAC 00, fs = 9584.640 Msps, f_fabric = 599.040 MHz, samps_per_clk = 16, type = axis_signal_gen_v6
Gen ch  1: DAC 01, fs = 9584.640 Msps, f_fabric = 599.040 MHz, samps_per_clk = 16, type = axis_signal_gen_v6
Gen ch  2: DAC 02, fs = 9584.640 Msps, f_fabric = 599.040 MHz, samps_per_clk = 16, type = axis_signal_gen_v6
Gen ch  3: DAC 03, fs = 9584.640 Msps, f_fabric = 599.040 MHz, samps_per_clk = 16, type = axis_signal_gen_v6
Gen ch  4: DAC 10, fs = 6881.280 Msps, f_fabric = 430.080 MHz, samps_per_clk = N/A, type = axis_sg_mixmux8_v1
Gen ch  5: DAC 11, fs = 6881.280 Msps, f_fabric = 430.080 MHz, samps_per_clk = 1, type = axis_sg_int4_v2
Gen ch  6: DAC 12, fs = 6881.280 Msps, f_fabric = 430.080 MHz, samps_per_clk = 1, type = axis_sg_int4_v2
Gen ch  7: DAC 13, fs = 6881.280 Msps, f_fabric = 430.080 MHz, samps_per_clk = 1, type = axis_sg_int4_v2
Gen ch  8: DAC 20, fs = 6881.280 Msps, f_fabric = 430.080 MHz, samps_per_clk = 1, type = axis_sg_int4_v2
Gen ch  9: DAC 21, fs = 6881.280 M

## 8.4 Readout effective sample rates

Each readout channel has an ADC. `f_output` is the decimated output rate that
determines the time resolution of acquired data.

In [27]:
for i, ro in enumerate(soccfg['readouts']):
    adcname = ro['adc']
    adc = soccfg['rf']['adcs'][adcname]
    print(f"Readout ch {i:2d}: ADC {adcname}, "
          f"fs = {adc['fs']:.3f} Msps, "
          f"f_output (decimated) = {ro['f_output']:.3f} MHz, "
          f"f_fabric = {ro['f_fabric']:.3f} MHz, "
          f"type = {ro['ro_type']}")

Readout ch  0: ADC 20, fs = 2457.600 Msps, f_output (decimated) = 307.200 MHz, f_fabric = 307.200 MHz, type = axis_dyn_readout_v1
Readout ch  1: ADC 22, fs = 2457.600 Msps, f_output (decimated) = 307.200 MHz, f_fabric = 307.200 MHz, type = axis_dyn_readout_v1
Readout ch  2: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.200 MHz, type = axis_pfb_readout_v4
Readout ch  3: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.200 MHz, type = axis_pfb_readout_v4
Readout ch  4: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.200 MHz, type = axis_pfb_readout_v4
Readout ch  5: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.200 MHz, type = axis_pfb_readout_v4
Readout ch  6: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.200 MHz, type = axis_pfb_readout_v4
Readout ch  7: ADC 21, fs = 2457.600 Msps, f_output (decimated) = 38.400 MHz, f_fabric = 307.20

## 8.5 `soc.get_sample_rates()` — live hardware rates

This queries the actual RFDC tile sample rates from the running hardware.

In [28]:
rates = soc.get_sample_rates()
print('Live sample rates from hardware:')
for k, v in rates.items():
    print(f'  {k}: {v}')

Live sample rates from hardware:
  dac: {0: 9584.64, 1: 6881.28, 2: 6881.28, 3: 6881.28}
  adc: {1: 2457.6, 2: 2457.6}


## 8.6 `soc.valid_sample_rates()` — allowed rates per tile

Shows what sample rates each DAC/ADC tile can be configured to.

In [29]:
# Valid sample rates for each DAC tile
print('=== Valid DAC sample rates ===')
for tile in range(4):
    try:
        valid = soc.valid_sample_rates('dac', tile)
        print(f'  DAC tile {tile}: {valid}')
    except Exception as e:
        print(f'  DAC tile {tile}: {e}')

print('\n=== Valid ADC sample rates ===')
for tile in range(4):
    try:
        valid = soc.valid_sample_rates('adc', tile)
        print(f'  ADC tile {tile}: {valid}')
    except Exception as e:
        print(f'  ADC tile {tile}: {e}')

=== Valid DAC sample rates ===
  DAC tile 0: [ 500.97230769  501.76        502.69090909  503.808       505.17333333
  506.88        510.42461538  512.          513.86181818  516.096
  518.82666667  519.87692308  522.24        522.24        525.03272727
  528.384       529.32923077  532.48        532.48        536.20363636
  537.6         540.672       542.72        546.13333333  547.37454545
  552.96        552.96        552.96        558.54545455  559.78666667
  561.73714286  563.2         565.248       568.32        569.71636364
  573.44        573.44        577.536       579.29142857  580.88727273
  583.68        587.09333333  589.824       592.05818182  596.84571429
  599.04        600.74666667  602.112       603.22909091  614.4
  614.4         614.4         614.4         614.4         625.57090909
  626.688       628.05333333  629.76        631.95428571  638.976
  641.70666667  645.12        649.50857143  651.264       655.36
  655.36        660.48        663.552       667.0628571

## 8.7 `soc.round_sample_rate()` — snap to nearest valid rate

Given a desired rate, returns the closest achievable sample rate.

In [30]:
# Round a desired sample rate to nearest valid value
for target in [5000, 7000, 10000]:
    rounded = soc.round_sample_rate(target, 'dac', 0)
    print(f'  DAC tile 0: requested {target} Msps -> nearest valid = {rounded} Msps')

RuntimeError: tiletype must be "dac" or "adc"

## 8.8 `soc.clocks_locked()` — check PLL lock status

In [ ]:
print('Clocks locked:', soc.clocks_locked())

---
# 9. Full raw config dump

Dump the entire `soccfg` dictionary for inspection.

In [ ]:
import json
print(json.dumps(soccfg.get_cfg(), indent=2, default=str))